Connecting to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Installing dependencies

In [ ]:
!pip -q install aiohttp

Script to download the guides

In [ ]:
%%writefile download_ifixit_guides.py
# --- BEGIN SCRIPT ---
import argparse
import asyncio
import json
import random
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Set

import aiohttp

API_BASE_DEFAULT = "https://www.ifixit.com/api/2.0"

@dataclass
class RetryConfig:
    max_attempts: int = 8
    base_delay: float = 0.75
    max_delay: float = 30.0
    jitter: float = 0.25

def _sleep_with_jitter(delay: float, jitter_frac: float) -> float:
    j = delay * jitter_frac
    return max(0.0, delay + random.uniform(-j, j))

async def fetch_json(session, url: str, retry: RetryConfig, sem: asyncio.Semaphore, timeout_s: int = 45) -> Dict[str, Any]:
    attempt = 0
    last_exc: Optional[BaseException] = None
    while attempt < retry.max_attempts:
        attempt += 1
        try:
            async with sem:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=timeout_s)) as resp:
                    if resp.status in (429, 500, 502, 503, 504):
                        ra = resp.headers.get("Retry-After")
                        if ra:
                            try:
                                wait = float(ra)
                            except ValueError:
                                wait = retry.base_delay * (2 ** (attempt - 1))
                        else:
                            wait = retry.base_delay * (2 ** (attempt - 1))
                        wait = min(wait, retry.max_delay)
                        await asyncio.sleep(_sleep_with_jitter(wait, retry.jitter))
                        continue
                    resp.raise_for_status()
                    return await resp.json(content_type=None)
        except (aiohttp.ClientError, asyncio.TimeoutError) as e:
            last_exc = e
            wait = min(retry.base_delay * (2 ** (attempt - 1)), retry.max_delay)
            await asyncio.sleep(_sleep_with_jitter(wait, retry.jitter))
    raise RuntimeError(f"Failed after {retry.max_attempts} attempts: {url}") from last_exc

def safe_get(d: Dict[str, Any], key: str, default=None):
    v = d.get(key, default)
    return v if v is not None else default

def flatten_guide(guide: Dict[str, Any]) -> Dict[str, Any]:
    gid = safe_get(guide, "guideid") or safe_get(guide, "id")
    title = safe_get(guide, "title")
    url = safe_get(guide, "url") or safe_get(guide, "web_url")
    intro = (safe_get(guide, "introduction_rendered") or safe_get(guide, "introduction_raw")
             or safe_get(guide, "introduction") or "")
    summary = safe_get(guide, "summary") or safe_get(guide, "description") or ""

    tools = []
    for t in safe_get(guide, "tools", []) or []:
        tools.append(safe_get(t, "name") or safe_get(t, "title") or str(t))

    parts = []
    for p in safe_get(guide, "parts", []) or []:
        parts.append(safe_get(p, "name") or safe_get(p, "title") or str(p))

    step_texts: List[str] = []
    for step in safe_get(guide, "steps", []) or []:
        lines = safe_get(step, "lines", None)
        if isinstance(lines, list) and lines:
            for ln in lines:
                txt = (safe_get(ln, "text_rendered") or safe_get(ln, "text_raw")
                       or safe_get(ln, "text") or "")
                if txt:
                    step_texts.append(txt)
        else:
            txt = (safe_get(step, "body_rendered") or safe_get(step, "body_raw")
                   or safe_get(step, "text_rendered") or safe_get(step, "text_raw")
                   or safe_get(step, "text") or "")
            if txt:
                step_texts.append(txt)

    device = safe_get(guide, "device") or safe_get(guide, "device_title") or safe_get(guide, "category") or ""
    locale = safe_get(guide, "locale") or safe_get(guide, "langid") or ""

    return {
        "guide_id": gid,
        "title": title,
        "url": url,
        "device": device,
        "locale": locale,
        "summary": summary,
        "introduction": intro,
        "tools": tools,
        "parts": parts,
        "steps": step_texts,
        "num_steps": len(safe_get(guide, "steps", []) or []),
        "fetched_at_epoch": int(time.time()),
    }

async def list_guide_ids(session, api_base: str, target: int, page_limit: int, start_offset: int,
                        retry: RetryConfig, sem: asyncio.Semaphore) -> List[int]:
    ids: List[int] = []
    offset = start_offset
    while len(ids) < target:
        url = f"{api_base}/guides?limit={page_limit}&offset={offset}"
        data = await fetch_json(session, url, retry, sem)
        page = data if isinstance(data, list) else (data.get("results") or data.get("guides") or [])
        if not page:
            break
        for item in page:
            gid = safe_get(item, "guideid") or safe_get(item, "id")
            if gid is None:
                continue
            try:
                ids.append(int(gid))
            except ValueError:
                continue
            if len(ids) >= target:
                break
        offset += page_limit
    return ids

async def download_guides(api_base: str, out_dir: Path, target_guides: int, concurrency: int,
                          page_limit: int, start_offset: int, resume: bool, sleep_between_requests: float):
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_dir = out_dir / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)

    seen_path = out_dir / "seen_ids.txt"
    jsonl_path = out_dir / "guides.jsonl"
    ids_path = out_dir / "guide_ids.txt"
    err_path = out_dir / "errors.log"

    seen: Set[int] = set()
    if resume and seen_path.exists():
        for line in seen_path.read_text().splitlines():
            line = line.strip()
            if line:
                try:
                    seen.add(int(line))
                except ValueError:
                    pass

    retry = RetryConfig()
    sem = asyncio.Semaphore(concurrency)

    headers = {"User-Agent": "ifixit-rag-downloader/1.0 (research)"}
    async with aiohttp.ClientSession(headers=headers) as session:
        collect_target = target_guides + (len(seen) if resume else 0)
        guide_ids = await list_guide_ids(session, api_base, collect_target, page_limit, start_offset, retry, sem)
        ids_path.write_text("\n".join(str(x) for x in guide_ids) + "\n")

        to_fetch = [gid for gid in guide_ids if gid not in seen]
        remaining_needed = max(0, target_guides - len(seen))
        to_fetch = to_fetch[:remaining_needed]

        print(f"Seen already: {len(seen)}")
        print(f"Will fetch now: {len(to_fetch)} (target total = {target_guides})")

        with open(jsonl_path, "a", encoding="utf-8") as jsonl_f:
            write_lock = asyncio.Lock()

            async def fetch_one(gid: int):
                url = f"{api_base}/guides/{gid}"
                guide = await fetch_json(session, url, retry, sem)

                (raw_dir / f"{gid}.json").write_text(json.dumps(guide, ensure_ascii=False), encoding="utf-8")
                flat = flatten_guide(guide)
                jsonl_f.write(json.dumps(flat, ensure_ascii=False) + "\n")
                jsonl_f.flush()

                async with write_lock:
                    with open(seen_path, "a", encoding="utf-8") as sf:
                        sf.write(str(gid) + "\n")

                if sleep_between_requests > 0:
                    await asyncio.sleep(sleep_between_requests)

            q: asyncio.Queue = asyncio.Queue()
            for gid in to_fetch:
                q.put_nowait(gid)

            completed = 0
            total = len(to_fetch)

            async def worker():
                nonlocal completed
                while True:
                    gid = await q.get()
                    if gid is None:
                        q.task_done()
                        return
                    try:
                        await fetch_one(gid)
                    except Exception as e:
                        with open(err_path, "a", encoding="utf-8") as ef:
                            ef.write(f"{gid}\t{repr(e)}\n")
                    finally:
                        completed += 1
                        if completed % 100 == 0 or completed == total:
                            print(f"Progress: {completed}/{total}")
                        q.task_done()

            workers = [asyncio.create_task(worker()) for _ in range(concurrency)]
            for _ in workers:
                q.put_nowait(None)

            await q.join()
            for w in workers:
                await w

        print("Done.")

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--out", required=True)
    p.add_argument("--target", type=int, default=60000)
    p.add_argument("--concurrency", type=int, default=8)
    p.add_argument("--page-limit", type=int, default=200)
    p.add_argument("--start-offset", type=int, default=0)
    p.add_argument("--api-base", default=API_BASE_DEFAULT)
    p.add_argument("--resume", action="store_true")
    p.add_argument("--sleep", type=float, default=0.05)
    return p.parse_args()

def main():
    args = parse_args()
    asyncio.run(download_guides(
        api_base=args.api_base,
        out_dir=Path(args.out),
        target_guides=args.target,
        concurrency=args.concurrency,
        page_limit=args.page_limit,
        start_offset=args.start_offset,
        resume=args.resume,
        sleep_between_requests=args.sleep,
    ))

if __name__ == "__main__":
    main()

Overwriting download_ifixit_guides.py


In [ ]:
!python download_ifixit_guides.py --out "/content/drive/MyDrive/ifixit_dump" --target 60000 --concurrency 8 --page-limit 200 --sleep 0.05

Seen already: 0
Will fetch now: 59502 (target total = 60000)
Progress: 100/59502
Progress: 200/59502
Progress: 300/59502
Progress: 400/59502
Progress: 500/59502
Progress: 600/59502
Progress: 700/59502
Progress: 800/59502
Progress: 900/59502
Progress: 1000/59502
Progress: 1100/59502
Progress: 1200/59502
Progress: 1300/59502
Progress: 1400/59502
Progress: 1500/59502
Progress: 1600/59502
Progress: 1700/59502
Progress: 1800/59502
Progress: 1900/59502
Progress: 2000/59502
Progress: 2100/59502
Progress: 2200/59502
Progress: 2300/59502
Progress: 2400/59502
Progress: 2500/59502
Progress: 2600/59502
Progress: 2700/59502
Progress: 2800/59502
Progress: 2900/59502
Progress: 3000/59502
Progress: 3100/59502
Progress: 3200/59502
Progress: 3300/59502
Progress: 3400/59502
Progress: 3500/59502
Progress: 3600/59502
Progress: 3700/59502
Progress: 3800/59502
Progress: 3900/59502
Progress: 4000/59502
Progress: 4100/59502
Progress: 4200/59502
Progress: 4300/59502
Progress: 4400/59502
Progress: 4500/59502
Pro

In [5]:
!wc -l "/content/drive/MyDrive/ifixit_dump/seen_ids.txt"
!wc -l "/content/drive/MyDrive/ifixit_dump/guides.jsonl"

59405 /content/drive/MyDrive/ifixit_dump/seen_ids.txt
59405 /content/drive/MyDrive/ifixit_dump/guides.jsonl
